# SmolLM2-135M Memory Fusion — Sequential Acceptance Training v3

Sequentially replace Transformer attention layers with TinyCeNN Memory Fusion. Accepted checkpoints and in-progress training are persisted on Google Drive.

The final two blocks test only the last formally accepted snapshot with prompts and then publish that exact tested snapshot to Hugging Face.

In [ ]:
import os, sys, pathlib, subprocess, json, time
import torch

subprocess.run(['nvidia-smi'], check=False)
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if not REPO_DIR.exists():
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], check=True)

subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR),'transformers==4.57.6','datasets>=3,<5','huggingface_hub>=0.34,<2','pandas','matplotlib','safetensors'], check=True)

# TinyCeNN-LM uses a src/ package layout. Add it explicitly to this running kernel.
REPO_SRC = REPO_DIR / 'src'
if str(REPO_SRC) not in sys.path:
    sys.path.insert(0, str(REPO_SRC))

import tinycenn_lm
print('python:', sys.executable)
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('tinycenn_lm:', tinycenn_lm.__file__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/TinyCeNN-LM')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('persistent root:', DRIVE_ROOT)

## Settings

`RESUME=True` continues from the accepted/in-progress state already stored on Drive.

In [ ]:
BASE_MODEL='HuggingFaceTB/SmolLM2-135M'
MEMORY_RANK=64
FEATURE_DIM=32
CONTEXT_LENGTH=128
SEED=73
RESUME=True
STRICT_ACCEPTANCE=True
MIN_LAYER_STEPS=50
MAX_LAYER_STEPS=300
CHECK_EVERY=25
LAYER_LR=2e-4
TEACHER_ALPHA_START=0.90
TEACHER_ALPHA_END=0.0
ACCEPT_NMSE=0.20
ACCEPT_COSINE=0.90
ACCEPT_INCREMENTAL_DELTA_NLL=0.015
ACCEPT_CUMULATIVE_DELTA_NLL=0.05
CORE_O_TOKENS=50_000
NORM_TOKENS=50_000
FULL_TOKENS=100_000
MAX_RUNTIME_MINUTES=240
OUTPUT_DIR=DRIVE_ROOT/f'smollm2-memory-fusion-sequential-r{MEMORY_RANK}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('output:', OUTPUT_DIR)

## Train / resume

`current_layer_needs_more_training` is not a crash. Rerun this cell to continue the same persisted layer.

In [ ]:
cmd=[sys.executable,'-u',str(REPO_DIR/'scripts'/'train_smollm2_memory_fusion_sequential_v3.py'),'--base-model',BASE_MODEL,'--output-dir',str(OUTPUT_DIR),'--memory-rank',str(MEMORY_RANK),'--feature-dim',str(FEATURE_DIM),'--context-length',str(CONTEXT_LENGTH),'--seed',str(SEED),'--min-layer-steps',str(MIN_LAYER_STEPS),'--max-layer-steps',str(MAX_LAYER_STEPS),'--check-every',str(CHECK_EVERY),'--layer-lr',str(LAYER_LR),'--teacher-alpha-start',str(TEACHER_ALPHA_START),'--teacher-alpha-end',str(TEACHER_ALPHA_END),'--accept-nmse',str(ACCEPT_NMSE),'--accept-cosine',str(ACCEPT_COSINE),'--accept-incremental-delta-nll',str(ACCEPT_INCREMENTAL_DELTA_NLL),'--accept-cumulative-delta-nll',str(ACCEPT_CUMULATIVE_DELTA_NLL),'--core-o-tokens',str(CORE_O_TOKENS),'--norm-tokens',str(NORM_TOKENS),'--full-tokens',str(FULL_TOKENS),'--max-runtime-minutes',str(MAX_RUNTIME_MINUTES)]
cmd.append('--resume' if RESUME else '--no-resume')
cmd.append('--strict-acceptance' if STRICT_ACCEPTANCE else '--no-strict-acceptance')
log_path=OUTPUT_DIR/'last_colab_run_v3.log'
print(' '.join(cmd))
print('Log:', log_path)
env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'
with log_path.open('w',encoding='utf-8') as log:
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    assert p.stdout is not None
    for line in p.stdout:
        print(line,end=''); log.write(line); log.flush()
    rc=p.wait()
print('Process return code:',rc)
if rc!=0:
    err=OUTPUT_DIR/'sequential_last_error.json'
    if err.exists():
        print('--- SAVED SOFTWARE ERROR ---'); print(err.read_text())
    raise RuntimeError(f'Sequential trainer failed with return code {rc}. See {log_path}.')

## Current persistent status

In [ ]:
status_path=OUTPUT_DIR/'sequential_run_status.json'
progress_path=OUTPUT_DIR/'sequential_progress.json'
in_progress_path=OUTPUT_DIR/'sequential_in_progress.json'
for p in [status_path,progress_path,in_progress_path]:
    if p.exists():
        print('\n###',p.name)
        data=json.loads(p.read_text())
        print(json.dumps(data,indent=2)[:12000])
if progress_path.exists():
    progress=json.loads(progress_path.read_text())
    print('\nAccepted layers:',len(progress.get('accepted_layers',[])),progress.get('accepted_layers',[]))
if in_progress_path.exists():
    cur=json.loads(in_progress_path.read_text())
    print('Current layer:',cur.get('current_layer'),'rounds completed:',cur.get('rounds_completed'))

## Block 1 — Prompt test of the last accepted snapshot

This block deliberately excludes the partially trained current layer. It also self-recovers the `src/` import path after a fresh Colab runtime.

In [ ]:
# BLOCK 1 — Load ONLY the last formally ACCEPTED checkpoint and test generation.
import os, sys, gc, json, pathlib, subprocess, torch

REPO_DIR=pathlib.Path(globals().get('REPO_DIR','/content/TinyCeNN-LM'))
if not REPO_DIR.exists():
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)
REPO_SRC=REPO_DIR/'src'
if str(REPO_SRC) not in sys.path:
    sys.path.insert(0,str(REPO_SRC))
try:
    import tinycenn_lm
except ModuleNotFoundError:
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)
    if str(REPO_SRC) not in sys.path:
        sys.path.insert(0,str(REPO_SRC))
    import tinycenn_lm
print('✅ tinycenn_lm imported from:',tinycenn_lm.__file__)

BASE_MODEL=globals().get('BASE_MODEL','HuggingFaceTB/SmolLM2-135M')
MEMORY_RANK=int(globals().get('MEMORY_RANK',64))
FEATURE_DIM=int(globals().get('FEATURE_DIM',32))
CONTEXT_LENGTH=int(globals().get('CONTEXT_LENGTH',128))
if 'OUTPUT_DIR' not in globals():
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    DRIVE_ROOT=pathlib.Path('/content/drive/MyDrive/TinyCeNN-LM')
    OUTPUT_DIR=DRIVE_ROOT/f'smollm2-memory-fusion-sequential-r{MEMORY_RANK}'

from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.smollm2_memory_fusion import SmolMemoryFusionConfig, replace_attention_layers, structural_summary

accepted_pt=OUTPUT_DIR/'sequential_progress.pt'
if not accepted_pt.exists():
    raise FileNotFoundError(f'No accepted checkpoint found at {accepted_pt}. Run/resume training until at least one layer passes acceptance.')
accepted_payload=torch.load(accepted_pt,map_location='cpu',weights_only=False)
accepted_layers=[int(x) for x in accepted_payload.get('accepted_layers',[])]
if not accepted_layers:
    raise RuntimeError('No layer has passed the acceptance gate yet.')
if accepted_layers!=list(range(len(accepted_layers))):
    raise RuntimeError(f'Accepted layers are not a contiguous prefix: {accepted_layers}')

mf_config=SmolMemoryFusionConfig.from_dict(accepted_payload['config'])
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL,use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype).to(device)
model.eval(); model.config.use_cache=False
if hasattr(model,'generation_config'):
    model.generation_config.use_cache=False

PROMPTS=['Austria is a country in Central Europe. The capital of Austria is','Question: What is the capital of Austria?\nAnswer:','Paris is the capital of France. Vienna is the capital of','The largest planet in the Solar System is','2 + 2 =']
@torch.inference_mode()
def greedy_completion(m,prompt,max_new_tokens=24):
    enc=tokenizer(prompt,return_tensors='pt',truncation=True,max_length=max(8,CONTEXT_LENGTH-max_new_tokens))
    enc={k:v.to(device) for k,v in enc.items()}
    ids=m.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,use_cache=False,pad_token_id=tokenizer.eos_token_id,eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(ids[0],skip_special_tokens=True)

print('Generating BASELINE completions...')
baseline_outputs=[greedy_completion(model,p) for p in PROMPTS]
replace_attention_layers(model,mf_config,accepted_layers)
incompatible=model.load_state_dict(accepted_payload['attention_state'],strict=False)
accepted_prefixes=tuple(f'model.layers.{i}.self_attn.' for i in accepted_layers)
missing_accepted=[k for k in incompatible.missing_keys if k.startswith(accepted_prefixes)]
if missing_accepted:
    raise RuntimeError(f'Accepted checkpoint is incomplete: {missing_accepted[:10]}')
if incompatible.unexpected_keys:
    raise RuntimeError(f'Unexpected checkpoint keys: {incompatible.unexpected_keys[:10]}')
model.eval(); model.requires_grad_(False); model.config.use_cache=False
if hasattr(model,'generation_config'):
    model.generation_config.use_cache=False
print('\nLoaded accepted Memory Fusion layers:',accepted_layers)
print('Structure:',json.dumps(structural_summary(model),indent=2))
print('\nGenerating ACCEPTED Memory Fusion completions...')
memory_outputs=[greedy_completion(model,p) for p in PROMPTS]
for i,prompt in enumerate(PROMPTS,1):
    print('\n'+'='*100)
    print(f'PROMPT {i}: {prompt}')
    print('\nBASELINE:\n',baseline_outputs[i-1])
    print('\nMEMORY FUSION (accepted only):\n',memory_outputs[i-1])
accepted_reports=[r for r in accepted_payload.get('layer_reports',[]) if r.get('accepted')]
last_accepted_report=accepted_reports[-1] if accepted_reports else None
smoke_test={'status':'prompt_generation_completed','base_model':BASE_MODEL,'accepted_layers':accepted_layers,'memory_fusion_config':mf_config.to_dict(),'last_accepted_report':last_accepted_report,'tests':[{'prompt':p,'baseline':b,'memory_fusion':m} for p,b,m in zip(PROMPTS,baseline_outputs,memory_outputs)]}
(OUTPUT_DIR/'prompt_smoke_test.json').write_text(json.dumps(smoke_test,indent=2,ensure_ascii=False),encoding='utf-8')
MODEL_FOR_EXPORT=model
TOKENIZER_FOR_EXPORT=tokenizer
ACCEPTED_PAYLOAD_FOR_EXPORT=accepted_payload
ACCEPTED_CONFIG_FOR_EXPORT=mf_config
PROMPT_SMOKE_TEST_FOR_EXPORT=smoke_test
print('\n✅ Prompt smoke test completed. Accepted snapshot is ready for export.')
gc.collect()

## Block 2 — Save the tested accepted snapshot to Hugging Face

Add `HF_TOKEN` to Colab Secrets for non-interactive login. If it is absent, the cell opens Hugging Face notebook login.

In [ ]:
# BLOCK 2 — Publish the exact prompt-tested ACCEPTED hybrid snapshot.
import json, os, pathlib, shutil, sys, torch
from huggingface_hub import HfApi, login, notebook_login, get_token
from safetensors.torch import save_file
if 'MODEL_FOR_EXPORT' not in globals():
    raise RuntimeError('Run BLOCK 1 first so the accepted snapshot is loaded and prompt-tested.')
REPO_DIR=pathlib.Path(globals().get('REPO_DIR','/content/TinyCeNN-LM'))
REPO_SRC=REPO_DIR/'src'
if str(REPO_SRC) not in sys.path:
    sys.path.insert(0,str(REPO_SRC))

hf_token=os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token=userdata.get('HF_TOKEN')
    except Exception:
        hf_token=None
if hf_token:
    login(token=hf_token,add_to_git_credential=False)
else:
    print('HF_TOKEN not found in Colab Secrets. Opening Hugging Face login...')
    notebook_login()
    hf_token=get_token()
if not hf_token:
    raise RuntimeError('Hugging Face login did not provide a token.')
api=HfApi(token=hf_token)
me=api.whoami()
HF_OWNER=me['name']
HF_REPO_ID=f'{HF_OWNER}/SmolLM2-135M-MemoryFusion-Sequential-R{MEMORY_RANK}'
HF_PRIVATE=False
accepted_layers=[int(x) for x in ACCEPTED_PAYLOAD_FOR_EXPORT['accepted_layers']]
accepted_reports=[r for r in ACCEPTED_PAYLOAD_FOR_EXPORT.get('layer_reports',[]) if r.get('accepted')]
last=accepted_reports[-1] if accepted_reports else {}
num_layers=int(MODEL_FOR_EXPORT.config.num_hidden_layers)
print('Hugging Face repository:',HF_REPO_ID)
print('Accepted layers being published:',accepted_layers)
export_dir=pathlib.Path('/content/hf_memory_fusion_export')
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True)
full_state={k:v.detach().cpu().contiguous().clone() for k,v in MODEL_FOR_EXPORT.state_dict().items()}
save_file(full_state,str(export_dir/'model.safetensors'),metadata={'format':'pt'})
del full_state
TOKENIZER_FOR_EXPORT.save_pretrained(export_dir)
status={}
status_file=OUTPUT_DIR/'sequential_run_status.json'
if status_file.exists():
    status=json.loads(status_file.read_text(encoding='utf-8'))
metadata={'format_version':1,'architecture':'smollm2-memory-fusion-sequential-accepted-prefix','base_model':BASE_MODEL,'source_repository':'https://github.com/vtavakkoli/TinyCeNN-LM','accepted_layers':accepted_layers,'num_hidden_layers':num_layers,'memory_fusion':ACCEPTED_CONFIG_FOR_EXPORT.to_dict(),'last_accepted_report':last,'trainer_status_at_export':status,'important':'Only formally accepted layers are included. Any sequential_in_progress layer is excluded.'}
(export_dir/'tinycenn_model.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
for name in ['sequential_progress.json','sequential_run_status.json','prompt_smoke_test.json']:
    src_file=OUTPUT_DIR/name
    if src_file.exists():
        shutil.copy2(src_file,export_dir/name)
loader_py='''from __future__ import annotations
import json
import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.smollm2_memory_fusion import SmolMemoryFusionConfig, replace_attention_layers

def load_model(repo_id: str, token=None, device=None):
    device=torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)
    meta_path=hf_hub_download(repo_id,'tinycenn_model.json',token=token)
    with open(meta_path,encoding='utf-8') as f:
        meta=json.load(f)
    tokenizer=AutoTokenizer.from_pretrained(repo_id,token=token,use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token=tokenizer.eos_token
    model=AutoModelForCausalLM.from_pretrained(meta['base_model'],dtype=dtype)
    cfg=SmolMemoryFusionConfig.from_dict(meta['memory_fusion'])
    replace_attention_layers(model,cfg,meta['accepted_layers'])
    state_path=hf_hub_download(repo_id,'model.safetensors',token=token)
    state=load_file(state_path,device='cpu')
    model.load_state_dict(state,strict=True)
    model.to(device).eval(); model.requires_grad_(False); model.config.use_cache=False
    if hasattr(model,'generation_config'):
        model.generation_config.use_cache=False
    return model,tokenizer,meta
'''
(export_dir/'load_model.py').write_text(loader_py,encoding='utf-8')
(export_dir/'requirements.txt').write_text('torch\ntransformers==4.57.6\nhuggingface_hub>=0.34,<2\nsafetensors\ngit+https://github.com/vtavakkoli/TinyCeNN-LM.git\n',encoding='utf-8')
def fmt(v):
    return 'n/a' if v is None else f'{float(v):.6f}'
readme=f'''---
license: apache-2.0
base_model: {BASE_MODEL}
tags:
- smollm2
- cenn
- memory-fusion
- hybrid-attention
- research
---

# SmolLM2-135M Memory Fusion — Sequential Accepted Snapshot

Research checkpoint from TinyCeNN-LM's sequential Memory Fusion conversion of `{BASE_MODEL}`.

- Total decoder layers: **{num_layers}**
- Formally accepted Memory Fusion layers: **{accepted_layers}** ({len(accepted_layers)}/{num_layers})
- Remaining layers keep original Transformer attention.
- Memory rank: **{MEMORY_RANK}**
- Feature dimension: **{FEATURE_DIM}**
- `use_cache=False` is required.

## Last accepted gate

| Metric | Value |
|---|---:|
| Layer | {last.get('layer','n/a')} |
| NMSE | {fmt(last.get('nmse'))} |
| Cosine similarity | {fmt(last.get('cosine'))} |
| Probe NLL | {fmt(last.get('probe_nll'))} |
| Incremental ΔNLL | {fmt(last.get('incremental_delta_nll'))} |
| Cumulative ΔNLL | {fmt(last.get('cumulative_delta_nll'))} |

These are acceptance diagnostics, not a claim of general benchmark superiority. `prompt_smoke_test.json` contains the deterministic prompt comparison run before publication.

## Loading

Install `requirements.txt`, then use the included `load_model.py` custom loader.

Source: https://github.com/vtavakkoli/TinyCeNN-LM
'''
(export_dir/'README.md').write_text(readme,encoding='utf-8')
api.create_repo(repo_id=HF_REPO_ID,repo_type='model',private=HF_PRIVATE,exist_ok=True)
api.upload_folder(folder_path=str(export_dir),repo_id=HF_REPO_ID,repo_type='model',commit_message=f'Publish accepted Memory Fusion prefix {accepted_layers}')
print('\n✅ Published accepted snapshot successfully')
print('https://huggingface.co/'+HF_REPO_ID)
print('Accepted layers published:',accepted_layers)
print('In-progress layer intentionally excluded:',status.get('current_layer'))